In [30]:
import pandas as pd
import ast
import re

sampled_sentences = pd.read_csv("../01_data/annotations/sample_annotations.csv")
main_dataset = pd.read_csv("../01_data/main_dataset/parlspeech_dataset.csv")
parl_questions_df = pd.read_csv("../01_data/main_dataset/parliamentary_questions_df.csv")


In [35]:
from nltk.tokenize.punkt import PunktSentenceTokenizer, PunktParameters

punkt_param = PunktParameters()
abbreviations = ['hon', 'mr', 'mrs', 'dr', 'ms', 'sir', 'prof']
punkt_param.abbrev_types = set(abbreviations)
tokenizer = PunktSentenceTokenizer(punkt_param)

def clean_text(text):
    # return empty string if the text column is not a string
    if not isinstance(text, str):
        return ""

    # replace misencoded punctuation
    replacements = {
        "Â£": "£",
        "â€œ": "“",
        "â€": "”",
        "â€˜": "‘",
        "â€™": "’",
        "â€“": "–",
        "â€”": "—",
        "â€¦": "…",
        "â€": '"',
        "Ã©": "é",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # normalize whitespaces
    text = re.sub(r"\s+", " ", text)

    # remove leading and trailing whitespaces
    return text.strip()

def split_sentences(text, tokenizer):
    if not isinstance(text, str) or not text.strip():
        return []
    return tokenizer.tokenize(text)


main_dataset['clean_text'] = main_dataset['text'].apply(clean_text)
main_dataset['sentences'] = main_dataset['clean_text'].apply(
    lambda t: split_sentences(t, tokenizer)
)

main_dataset['sentences'] = main_dataset['sentences'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
sentences_df = main_dataset.explode('sentences').reset_index(drop=True)
sentences_df = sentences_df.rename(columns={'sentences': 'sentence'})

elections = pd.to_datetime(["2015-05-07", "2017-06-08", "2019-12-12"])
campaign_windows = pd.DataFrame({
    "start": elections - pd.DateOffset(months=2),
    "end": elections
})
main_dataset["date"] = pd.to_datetime(main_dataset["date"])
def in_window(x):
    return ((campaign_windows["start"] <= x) & (x <= campaign_windows["end"])).any()
main_dataset["in_campaign_period"] = main_dataset["date"].apply(in_window)
campaign_df = main_dataset[main_dataset["in_campaign_period"]]


In [101]:
# get the counts of a snipper per party
snippet = "old people"
snippet_sentences = campaign_df[campaign_df["sentence"].str.contains(snippet)]
snippet_sentences.groupby("party").size()

party
Con       2
DUP       1
Lab       7
LibDem    2
dtype: int64

In [32]:
parl_questions_df['sentences'] = parl_questions_df['sentences'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
sentences_df = parl_questions_df.explode('sentences').reset_index(drop=True)
sentences_df = sentences_df.rename(columns={'sentences': 'sentence'})
sentences_df.head()

KeyError: 'sentences'

In [12]:
sentences_per_party = sentences_df.groupby("party").size()
regex_pattern = re.compile(r".*\?$")
question_sentences = sentences_df[sentences_df["sentence"].str.match(regex_pattern)]
unique_speeches_df = question_sentences.drop_duplicates(subset=["date", 'speaker', 'speechnumber'])
all_question_sentences = pd.merge(sentences_df, unique_speeches_df[["date", "speaker", "speechnumber"]], on=["date", "speaker", "speechnumber"], how="inner")

In [13]:
elections = pd.to_datetime(["2015-05-07", "2017-06-08", "2019-12-12"])
campaign_windows = pd.DataFrame({
    "start": elections - pd.DateOffset(months=2),
    "end": elections
})
all_question_sentences["date"] = pd.to_datetime(all_question_sentences["date"])
def in_window(x):
    return ((campaign_windows["start"] <= x) & (x <= campaign_windows["end"])).any()
all_question_sentences["in_campaign_period"] = all_question_sentences["date"].apply(in_window)
campaign_df = all_question_sentences[all_question_sentences["in_campaign_period"]]

In [29]:
campaign_df.head()

,date,agenda,speechnumber,speaker,party,party.facts.id,chair,terms,text,parliament,iso3country,month,year,clean_text,sentence,in_campaign_period
96837,2015-03-09,Unemployment (North West Norfolk) [Oral Answer...,3,Henry Bellingham,Con,1567.0,False,52,That is encouraging and unemployment in my con...,UK-HouseOfCommons,GBR,3,2015,That is encouraging and unemployment in my con...,That is encouraging and unemployment in my con...,True
96838,2015-03-09,Unemployment (North West Norfolk) [Oral Answer...,3,Henry Bellingham,Con,1567.0,False,52,That is encouraging and unemployment in my con...,UK-HouseOfCommons,GBR,3,2015,That is encouraging and unemployment in my con...,Following the story in The Sunday Times last w...,True
96839,2015-03-09,Young People (Employment or Education) [Oral A...,8,Stephen Metcalfe,Con,1567.0,False,60,One barrier to young people seeking employment...,UK-HouseOfCommons,GBR,3,2015,One barrier to young people seeking employment...,One barrier to young people seeking employment...,True
96840,2015-03-09,Young People (Employment or Education) [Oral A...,8,Stephen Metcalfe,Con,1567.0,False,60,One barrier to young people seeking employment...,UK-HouseOfCommons,GBR,3,2015,One barrier to young people seeking employment...,Will my right hon. Friend work with colleagues...,True
96841,2015-03-09,Young People (Employment or Education) [Oral A...,10,Rehman Chishti,Con,1567.0,False,46,Will the Minister welcome the initiative that ...,UK-HouseOfCommons,GBR,3,2015,Will the Minister welcome the initiative that ...,Will the Minister welcome the initiative that ...,True


In [28]:
# get the counts of a snippet per party
snippet = "young"
snippet_sentences = campaign_df[campaign_df["sentence"].str.contains(snippet)]
snippet_sentences.groupby("party").size()

party
Con       34
DUP        2
Lab       31
LibDem     5
dtype: int64

In [57]:
# get the specific snippet
specific_snippet = "Despite all the controversy of the recent changes, more young students are applying to go to university than ever before, there is a higher rate of students from disadvantaged backgrounds going to university than ever before, and a higher proportion of youngsters from black and minority ethnic backgrounds are going to university than ever before, confounding all the predictions that the hon. Lady's party made at the time of the change."
sentences_df[sentences_df["clean_text"].str.contains(specific_snippet)]

,date,agenda,speechnumber,speaker,party,party.facts.id,chair,terms,text,parliament,iso3country,month,year,clean_text,sentence
230077,2014-10-14,Topical Questions [Oral Answers to Questions >...,73,David Cameron,Con,1567.0,False,96,It is worth remembering what is happening righ...,UK-HouseOfCommons,GBR,10,2014,It is worth remembering what is happening righ...,It is worth remembering what is happening righ...
230078,2014-10-14,Topical Questions [Oral Answers to Questions >...,73,David Cameron,Con,1567.0,False,96,It is worth remembering what is happening righ...,UK-HouseOfCommons,GBR,10,2014,It is worth remembering what is happening righ...,Despite all the controversy of the recent chan...
230079,2014-10-14,Topical Questions [Oral Answers to Questions >...,73,David Cameron,Con,1567.0,False,96,It is worth remembering what is happening righ...,UK-HouseOfCommons,GBR,10,2014,It is worth remembering what is happening righ...,I suspect that the effects of individual voter...


In [16]:
print(len(sentences_df))
print(len(question_sentences))
print(len(all_question_sentences))

514236
91103
200202


In [18]:
sentences_per_party = all_question_sentences.groupby("party").size()
sentences_per_party

party
APNI                           157
Birkenhead Social Justice        6
Change UK                       57
Con                          76825
DUP                           3137
GPEW                           416
Independent                    445
Lab                          95204
LibDem                        9933
PlaidCymru                    1244
Respect                          9
SDLP                           696
SNP                          11704
The Independents                11
UKIP                           122
UUP                            209
dtype: int64

In [34]:
questions_per_party = all_question_sentences.groupby("party").size()
questions_per_party

party
APNI                           157
Birkenhead Social Justice        6
Change UK                       57
Con                          76825
DUP                           3137
GPEW                           416
Independent                    445
Lab                          95204
LibDem                        9933
PlaidCymru                    1244
Respect                          9
SDLP                           696
SNP                          11704
The Independents                11
UKIP                           122
UUP                            209
dtype: int64

In [39]:
# print some example questions
for idx in range(20):
    print(f"Question from party {question_sentences["party"].iloc[idx]}")
    #print("-"*100)
    print(question_sentences["sentence"].iloc[idx])
    print("-"*100)

Question from party Con
May I take this opportunity to welcome my right hon. Friend to the Dispatch Box and to congratulate him on his new and important role?
----------------------------------------------------------------------------------------------------
Question from party Con
Will he reassure the House and my constituents that value for money will be at the heart of his Department's vital work in tackling poverty in the poorest countries in the world?
----------------------------------------------------------------------------------------------------
Question from party Lab
May I ask whether he regards educating young girls in Afghanistan as a valuable part of that comprehensive approach or whether he agrees with the Defence Secretary that it is simply“education policy in a broken 13th-century country”?
----------------------------------------------------------------------------------------------------
Question from party Lab
Has he also secured the re-education of the new Secre